# Word count CORD-19 distribuito su CloudVeneto - Giulia

Questo notebook e' la versione distribuita del word count locale e vive nella cartella `Giulia/` della repo condivisa.

Va eseguito **solo su CloudVeneto**, dentro `/data/MAPD-Project`, dopo aver verificato che i dati puliti siano disponibili in:

```text
/data/MAPD-Project/data/silver/paragraphs
```

La configurazione usa le informazioni ricostruite durante il setup:

| Ruolo | Nome VM | IP privato | Uso Dask |
|---|---|---:|---|
| scheduler / notebook | MAPD-project-1 | 10.67.22.118 | scheduler, non worker di default |
| worker | MAPD-project-2 | 10.67.22.206 | 1 worker, 1 thread |
| worker | MAPD-project-3 | 10.67.22.53 | 1 worker, 1 thread |

La VM scheduler viene lasciata libera di default per evitare instabilita': ci girano gia' notebook, scheduler, SSH e sistema operativo. Se serve provare anche 3 worker, puoi impostare `USE_SCHEDULER_AS_WORKER = True`, ma prima fai un run con il default.

## 1. Configurazione cluster e parametri

I parametri sono volutamente prudenti per VM `cldareapd.medium`:

- `1` processo worker per VM;
- `1` thread per worker, cosi' evitiamo picchi di memoria e contesa GIL;
- `2500MiB` per worker, lasciando RAM a OS, SSH, scheduler e notebook;
- spill Dask su `/data/dask-worker-space`, quindi sul volume dati;
- `split_out=12`, abbastanza per distribuire lo shuffle su 2 worker senza creare troppe partizioni intermedie.

In [ ]:
from pathlib import Path
import os
import shlex
import socket
import subprocess
import sys

ROOT = Path("/data/MAPD-Project")
GIULIA_ROOT = ROOT / "Giulia"
if not ROOT.exists():
    raise RuntimeError(
        "Questo notebook va eseguito su CloudVeneto, non sul Mac. "
        "Aprilo da VS Code connesso alla VM e dalla cartella /data/MAPD-Project."
    )

os.chdir(ROOT)

SCHEDULER_IP = "10.67.22.118"
WORKER_IPS = ["10.67.22.206", "10.67.22.53"]
USE_SCHEDULER_AS_WORKER = False

WORKER_MEMORY = "2500MiB"
THREADS_PER_WORKER = 1
SPLIT_OUT = 12
TOP_N = 50
RUN_NAME = "giulia_word_count_cloudveneto"

DATA = ROOT / "data"
INPUT = DATA / "silver" / "paragraphs"
OUT = GIULIA_ROOT / "reports" / RUN_NAME

print("host        :", socket.gethostname())
print("python      :", sys.executable)
print("root        :", ROOT)
print("giulia root :", GIULIA_ROOT)
print("input       :", INPUT)
print("output      :", OUT)
print("scheduler   :", SCHEDULER_IP)
print("workers     :", WORKER_IPS)
print("worker mem  :", WORKER_MEMORY)
print("split_out   :", SPLIT_OUT)

## 2. Check preliminare dei dati e dello script

Questa cella controlla che il notebook stia vedendo i dati puliti e lo script distribuito.

Non carica il dataset in memoria: legge solo i metadati Parquet.

In [ ]:
import pyarrow.dataset as ds

script_path = GIULIA_ROOT / "scripts" / "run_word_count_cloudveneto.py"
if not script_path.exists():
    raise FileNotFoundError(f"Script mancante: {script_path}")
if not INPUT.exists():
    raise FileNotFoundError(
        f"Dataset pulito non trovato: {INPUT}
"
        "Controlla che il volume /data sia montato e che i worker lo vedano."
    )

dataset = ds.dataset(INPUT, format="parquet")
print(dataset.schema)
print("script ok:", script_path)

## 3. Run distribuito completo

Questa cella avvia un `SSHCluster`, aspetta i worker, verifica che ogni worker veda `/data/MAPD-Project/data/silver/paragraphs`, poi lancia il MapReduce completo tramite `scripts/word_count_dask.py`.

Output attesi in:

```text
/data/MAPD-Project/Giulia/reports/giulia_word_count_cloudveneto/
```

File principali:

- `global_word_counts/`, Parquet completo con conteggi globali;
- `top_words.csv`, top parole;
- `top_words.png`, grafico;
- `summary.json`, riassunto;
- `benchmark.json`, tempi e configurazione cluster.

In [ ]:
cmd = [
    sys.executable,
    str(GIULIA_ROOT / "scripts" / "run_word_count_cloudveneto.py"),
    "--project-root", str(ROOT),
    "--scheduler-ip", SCHEDULER_IP,
    "--worker-ips", ",".join(WORKER_IPS),
    "--worker-memory", WORKER_MEMORY,
    "--threads-per-worker", str(THREADS_PER_WORKER),
    "--split-out", str(SPLIT_OUT),
    "--top-n", str(TOP_N),
    "--run-name", RUN_NAME,
]

if USE_SCHEDULER_AS_WORKER:
    cmd.append("--use-scheduler-as-worker")

print("Comando:")
print(" ".join(shlex.quote(part) for part in cmd))

subprocess.run(cmd, check=True)

## 4. Lettura dei risultati

Questa cella legge solo gli output finali, quindi e' leggera. Usala per verificare che il run sia riuscito e per prendere numeri da riportare nella relazione.

In [ ]:
import json
import pandas as pd

summary_path = OUT / "summary.json"
benchmark_path = OUT / "benchmark.json"
top_csv = OUT / "top_words.csv"

if not summary_path.exists():
    raise FileNotFoundError(f"summary non trovato: {summary_path}. Prima esegui la cella 3.")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
top_words = pd.read_csv(top_csv)

print("total seconds:", summary.get("total_seconds"))
print("total tokens :", summary.get("total_tokens"))
print("vocab size   :", summary.get("vocabulary_size"))
print("cluster      :", benchmark.get("cluster"))
print("timings      :", benchmark.get("timings"))

top_words.head(20)

## 5. Note per benchmark e stabilita'

Per confrontare run diversi, cambia **una cosa alla volta**:

- `WORKER_IPS = ["10.67.22.206"]` per benchmark a 1 worker;
- `WORKER_IPS = ["10.67.22.206", "10.67.22.53"]` per benchmark a 2 worker;
- `USE_SCHEDULER_AS_WORKER = True` solo se serve un test a 3 worker, ma e' meno stabile.

Mantieni `THREADS_PER_WORKER = 1` e non abbassare troppo `WORKER_MEMORY`: con meno memoria Dask passa piu' tempo a fare pause/spill che a lavorare.

Se un worker dice che non vede l'input, esegui sullo scheduler:

```bash
cd /data/MAPD-Project
bash scripts/cluster_storage_up.sh 10.67.22.206 10.67.22.53
```

Poi rilancia questo notebook.